# 14. 11b (LLM) + 13b (KoCharELECTRA) 합치기
| 단계 | 문맥 판단 | 재료 | 검증 F1 |
|---|---|---|---|
| 11b | 미세조정 Qwen 1.5B | 생성 문장으로 넓힌 01, v2·v3 태거 | 0.9915 |
| 13b | 글자 언어모델 + 5-gram + KoCharELECTRA discriminator | 원래 01, v2 태거 | 0.9922 |
| **14** | 위 전부 | 11b 의 재료 | ? |

13b 와 11b 는 평균은 비슷하지만 서로 다른 문장을 맞혔다 (13b 가 나은 문장 153개, 11b 가 나은 문장 136개). 또 11b 의 재료는 고정 어절 정확도(99.12%)와 후보 상한(0.9963)이 13b 의 재료(98.75%, 0.9950)보다 높다.

점수 = 채널 유사도 + w_llm × Qwen + w × 글자 언어모델 + w_kn × 5-gram + w_rtd × discriminator

**전제 (모두 드라이브에 있음):** `outputs10/` 의 11b 결과와 후보 캐시, `outputs/lm_adapter_split|full`, `outputs/charlm_best.pt`·`charlm_full.pt` (30 epoch), `outputs/gen_reviews_full.txt`.
**GPU 런타임 (A100 권장: Qwen 이 계산의 대부분), 첫 셀부터 차례로. A100 에서 전체 약 1시간.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' --exclude 'models' "$DRIVE/" /content/work/
%cd /content/work
!ls -d outputs10/anchor_items_val_agree_v2-v3_k20_b64_uo.pkl outputs10/anchor_items_test_agree_v2-v3_k20_b64_uo.pkl outputs10/submission_noisy_channel.csv outputs/lm_adapter_split outputs/lm_adapter_full outputs/charlm_best.pt outputs/charlm_full.pt outputs/gen_reviews_full.txt 2>&1; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
for f in ('charlm_best', 'charlm_full'):
    print(f, 'epoch', torch.load(f'outputs/{f}.pt', map_location='cpu')['epoch'])
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    !mkdir -p "$DRIVE/outputs10" && rsync -a --update /content/work/outputs10/ "$DRIVE/outputs10/"
    print('saved outputs, outputs10 -> drive')

## 1) 미세조정 Qwen 조립 (저장된 어댑터로, 1~2분)

In [ ]:
!pip -q install peft
!pip -q uninstall -y torchao
!python -u lm_finetune.py --merge_only

## 2) 검증 + test 예측 (한 번에)
먼저 검증에서 가중치 18조합을 탐색하고, 이어서 그 최고 조합으로 test 를 예측해 제출 파일까지 만든다. 여기서 새로 학습하는 모델은 없다 (모두 저장된 모델을 불러와 채점만 한다).

탐색: w_llm (0, 0.5, 1.0) × 12b 항 (끔 / 글자 언어모델 0.5 + 5-gram 0.5) × w_rtd (0, 0.5, 1.0).

검증 출력에서 확인할 줄:
- `w_llm 1.0  w 0  w_kn 0  w_rtd 0` = 11b 재현 → **0.9915** 여야 정상
- `w_llm 0.0  w 0.5  w_kn 0.5  w_rtd 0.5~1.0` = 11b 의 재료 위에서 13b 방식 (Qwen 채점 없음)
- 그 밖의 줄이 둘 다보다 높으면 합친 효과가 있는 것

검증이 끝나면 한 번 저장하므로, test 도중에 끊겨도 검증 결과는 드라이브에 남는다.
결과: `outputs10/val_pred_anchor_agree_v2-v3_k20_b64_uo_14.csv`, `best_anchor_..._14.json`, `outputs10/submission_anchor_agree_v2-v3_k20_b64_uo_14.csv`, 기록 `outputs10/anchor_runs.csv` (name = 14).

In [ ]:
!python -u anchor_combo14.py --mode val --llm_weights 0,0.5,1.0 --base_weights 0:0,0.5:0.5 --rtd_weights 0,0.5,1.0 --batch_tokens 32768
save_back()
!python -u lm_finetune.py --full --merge_only
!python -u anchor_combo14.py --mode test --aug_file outputs/gen_reviews_full.txt --batch_tokens 32768
save_back()

## 3) 결과 비교
최고 가중치가 탐색 범위 끝에 걸렸는지, 13b (0.9922) 보다 높은지 확인. 13b 보다 낮으면 제출은 13b 파일(`outputs/submission_anchor_agree_v2_k20_b64_uo_13b.csv`)을 그대로 쓴다.

In [ ]:
import json
import pandas as pd
a = pd.read_csv('outputs10/anchor_runs.csv'); a['name'] = a['name'].astype(str)
g = a[a.name == '14'].rename(columns={'w': 'w_llm'})
print(g.sort_values('f1', ascending=False)[['w_llm', 'w_nn', 'w_kn', 'w_rtd', 'f1', 'sec']].to_string())
B = json.load(open('outputs10/best_anchor_agree_v2-v3_k20_b64_uo_14.json', encoding='utf-8'))
print({k: B[k] for k in ('w_llm', 'w', 'w_kn', 'w_rtd', 'f1')})
v14 = pd.read_csv('outputs10/val_pred_anchor_agree_v2-v3_k20_b64_uo_14.csv').set_index('ID')
for nm, f in (('11b', 'outputs10/val_pred_anchor_agree_v2-v3_k20_b64_uo_11b.csv'), ('13b', 'outputs/val_pred_anchor_agree_v2_k20_b64_uo_13b.csv')):
    v = pd.read_csv(f).set_index('ID'); d = v14.f1 - v.f1.loc[v14.index]
    print(f'14 {v14.f1.mean():.4f} vs {nm} {v.f1.mean():.4f} | 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}')
t = pd.read_csv('data/test.csv'); s = pd.read_csv('outputs10/submission_anchor_agree_v2-v3_k20_b64_uo_14.csv')
print('제출 파일:', len(s), '행 | ID 일치', (t.ID == s.ID).all(), '| 빈 값', s.output.isna().sum(), '| 길이 일치', (t.input.str.len() == s.output.str.len()).mean())

### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()